In [3]:
import pandas as pd
import numpy as np
import os

def haversine(lat1, lon1, lat2, lon2):
    """Calculates physical distance in km between two GPS points."""
    R = 6371.0 # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

print("🚀 Step 1: Loading Node Metadata...")

# Pointing two folders up (../../) from src/data/ to the root project folder
# Using the 'r' before the string ensures Windows backslashes are read correctly
metadata_path = r"c:\Users\ANISH\Desktop\FRP\ecstgnn\ec-st-gnn-airpollution-forecasting\data\processed\node_metadata.csv"
output_matrix_path = r"c:\Users\ANISH\Desktop\FRP\ecstgnn\ec-st-gnn-airpollution-forecasting\data\processed\adjacency_matrix.npy"

nodes = pd.read_csv(metadata_path)

# --- Map String Names to Numeric IDs ---
stations = nodes['station'].unique()
station_to_idx = {s: i for i, s in enumerate(stations)}
nodes['station_id'] = nodes['station'].map(station_to_idx)

# Save the updated nodes file permanently
nodes.to_csv(metadata_path, index=False)
print("✅ Node IDs mapped successfully!")

# --- K-NN Graph + Self Loops ---
print("🚀 Step 2: Calculating Distances & Building K-NN Graph...")
n = len(nodes)
distances = np.zeros((n, n))
adj_matrix = np.zeros((n, n))
k_neighbors = 5 # Each station connects to its 5 closest peers

# Calculate all distances
for i in range(n):
    for j in range(n):
        if i != j:
            distances[i, j] = haversine(
                nodes.iloc[i]['latitude'], nodes.iloc[i]['longitude'],
                nodes.iloc[j]['latitude'], nodes.iloc[j]['longitude']
            )

# Build Adjacency Matrix using K-NN
for i in range(n):
    # argsort sorts distances; [1:k+1] grabs the closest K (skipping 0, which is itself)
    closest_indices = np.argsort(distances[i])[1:k_neighbors+1]
    for j in closest_indices:
        adj_matrix[i, j] = 1.0

# 🔥 CRITICAL GNN FIX: Add Self-Loops 🔥
np.fill_diagonal(adj_matrix, 1.0)

print("\nFinal Adjacency Matrix (Top Left 5x5):")
print(adj_matrix[:5, :5])

# Save the Matrix
np.save(output_matrix_path, adj_matrix)
print(f"\n✅ SUCCESS: adjacency_matrix.npy created at {output_matrix_path}")

🚀 Step 1: Loading Node Metadata...
✅ Node IDs mapped successfully!
🚀 Step 2: Calculating Distances & Building K-NN Graph...

Final Adjacency Matrix (Top Left 5x5):
[[1. 0. 0. 0. 1.]
 [0. 1. 1. 0. 0.]
 [0. 1. 1. 0. 1.]
 [0. 1. 1. 1. 0.]
 [1. 0. 0. 0. 1.]]

✅ SUCCESS: adjacency_matrix.npy created at c:\Users\ANISH\Desktop\FRP\ecstgnn\ec-st-gnn-airpollution-forecasting\data\processed\adjacency_matrix.npy
